<div style="text-align:left;">
    <span style="
        display:inline-block;
        background-color:#4169E1;
        color:white;
        padding:10px 20px;
        border-radius:8px;
        font-size:45px;
        font-weight:bold;
    ">
        AutoML for EHR Mortality Risk Prediction
    </span>
</div>

**Author:** Sarra Chouk 

**Student ID:** 60300372

**Project:** EHR Mortality Risk Prediction  

**Dataset:** MIMIC-IV

**Date:** April 4, 2026  

---

### **Objective**
To evaluate an AutoML pipeline on the same engineered dataset used in the manual modeling pipeline.

### **Setup and Imports**

In [1]:
import os
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    f1_score,
    precision_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

import plotly.graph_objects as go
import plotly.express as px

from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")

SEED = 42
TARGET_COL = "target"

TRACK = "raw"       # "raw" or "imputed"
RUN_MODE = "core"   # "core" or "full"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = DATA_ROOT / ("splits" if TRACK == "raw" else "splits_imputed")

AUTOML_ROOT = PROJECT_ROOT / "models" / "automl"
AUTOML_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"autogluon_{TRACK}_{RUN_MODE}"
SAVE_PATH = AUTOML_ROOT / RUN_NAME

DROP_FEATURE_COLS = ["target", "subject_id", "hadm_id", "admittime", "_original_order"]

VAL_SIZE = 0.20
BUSINESS_MIN_PRECISION = None

COLORS = {
    "primary": "#3146D3",
    "danger": "#EE3158",
    "warning": "#FFA800",
    "accent": "#1CBFC1"
}

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)
print("SPLIT_DIR exists:", SPLIT_DIR.exists())
print("TRACK:", TRACK)
print("RUN_MODE:", RUN_MODE)
print("SAVE_PATH:", SAVE_PATH)

NOTEBOOK_DIR: /Users/sarrachouk/Desktop/ehr-mortality-prediction/notebooks
PROJECT_ROOT: /Users/sarrachouk/Desktop/ehr-mortality-prediction
SPLIT_DIR: /Users/sarrachouk/Desktop/ehr-mortality-prediction/data/processed/splits
SPLIT_DIR exists: True
TRACK: raw
RUN_MODE: core
SAVE_PATH: /Users/sarrachouk/Desktop/ehr-mortality-prediction/models/automl/autogluon_raw_core


/opt/anaconda3/envs/aba/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Helper Functions**

In [2]:
def auto_find_full_split_files(split_dir: Path):
    if not split_dir.exists():
        raise FileNotFoundError(f"Split directory does not exist: {split_dir}")

    files = {p.name.lower(): p for p in split_dir.glob("*.csv")}
    print("Detected files:", sorted(files.keys()))

    def find_one(candidates):
        for name, path in files.items():
            for c in candidates:
                if c == name or c in name:
                    return path
        return None

    paths = {
        "train_full": find_one(["train_full.csv", "train_full"]),
        "test_full": find_one(["test_full.csv", "test_full"]),
        "deployment_full": find_one(["deployment_full.csv", "deployment_full"]),
    }

    missing = [k for k, v in paths.items() if v is None]
    if missing:
        raise ValueError(f"Missing required full split files: {missing}")

    return paths


def check_required_columns(df, required_cols):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "threshold": threshold
    }
    return metrics


def threshold_search_table(y_true, y_prob):
    thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
    rows = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rows.append({
            "threshold": thr,
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred)
        })
    return pd.DataFrame(rows)


def choose_threshold_from_validation(threshold_df, min_precision=None):
    df = threshold_df.copy()

    if min_precision is not None:
        filtered = df[df["precision"] >= min_precision].copy()
        if len(filtered) > 0:
            df = filtered

    df = df.sort_values(
        by=["recall", "f1", "precision", "threshold"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    return df.iloc[0]["threshold"], df


def get_positive_class_proba(predictor, data, model=None):
    proba = predictor.predict_proba(data, model=model, as_pandas=True)

    if isinstance(proba, pd.DataFrame):
        if 1 in proba.columns:
            return proba[1].values
        elif "1" in proba.columns:
            return proba["1"].values
        else:
            return proba.iloc[:, -1].values
    return proba.values


def evaluate_split(predictor, X, y, threshold, split_name="split", model=None):
    y_prob = get_positive_class_proba(predictor, X, model=model)
    metrics = compute_binary_metrics(y, y_prob, threshold)
    metrics["split"] = split_name
    metrics["model"] = model if model is not None else predictor.model_best
    return metrics, y_prob


def rank_candidate_models(candidate_results_df):
    return candidate_results_df.sort_values(
        by=["pr_auc", "recall", "f1"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

### **Plotting Functions**

In [3]:
def plot_pr_curve_plotly(y_true, y_prob, title):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        mode="lines",
        name=f"PR Curve (AP={ap:.4f})",
        line=dict(color=COLORS["primary"], width=3)
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Recall",
        yaxis_title="Precision",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()


def plot_roc_curve_plotly(y_true, y_prob, title):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"ROC Curve (AUC={auc:.4f})",
        line=dict(color=COLORS["accent"], width=3)
    ))

    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random",
        line=dict(color="gray", dash="dash")
    ))

    fig.update_layout(
        title=title,
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()


def plot_threshold_analysis_plotly(threshold_df, title="Threshold vs Metrics"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=threshold_df["threshold"],
        y=threshold_df["recall"],
        mode="lines",
        name="Recall",
        line=dict(color=COLORS["danger"], width=3)
    ))

    fig.add_trace(go.Scatter(
        x=threshold_df["threshold"],
        y=threshold_df["f1"],
        mode="lines",
        name="F1 Score",
        line=dict(color=COLORS["warning"], width=3)
    ))

    fig.add_trace(go.Scatter(
        x=threshold_df["threshold"],
        y=threshold_df["precision"],
        mode="lines",
        name="Precision",
        line=dict(color=COLORS["accent"], width=3)
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Threshold",
        yaxis_title="Score",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()


def show_confusion_plotly(y_true, y_prob, threshold, title):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    fig = px.imshow(
        cm,
        text_auto=True,
        color_continuous_scale=[
            [0.0, "#ffffff"],
            [1.0, COLORS["primary"]]
        ],
        x=["Pred 0", "Pred 1"],
        y=["True 0", "True 1"],
        labels=dict(x="Predicted", y="Actual", color="Count")
    )
    fig.update_layout(title=title)
    fig.show()

### **Load the Engineered Split Files**

In [4]:
split_files = auto_find_full_split_files(SPLIT_DIR)
split_files

Detected files: ['deployment_full.csv', 'test_full.csv', 'train_full.csv', 'x_deployment.csv', 'x_test.csv', 'x_train.csv', 'y_deployment.csv', 'y_test.csv', 'y_train.csv']


{'train_full': PosixPath('/Users/sarrachouk/Desktop/ehr-mortality-prediction/data/processed/splits/train_full.csv'),
 'test_full': PosixPath('/Users/sarrachouk/Desktop/ehr-mortality-prediction/data/processed/splits/test_full.csv'),
 'deployment_full': PosixPath('/Users/sarrachouk/Desktop/ehr-mortality-prediction/data/processed/splits/deployment_full.csv')}

In [5]:
train_full = pd.read_csv(split_files["train_full"])
test_full = pd.read_csv(split_files["test_full"])
deployment_full = pd.read_csv(split_files["deployment_full"])

print("train_full:", train_full.shape)
print("test_full:", test_full.shape)
print("deployment_full:", deployment_full.shape)

train_full: (380461, 54)
test_full: (83356, 54)
deployment_full: (82211, 54)


### **Sanity Checks**

In [6]:
check_required_columns(train_full, ["subject_id", "target"])
check_required_columns(test_full, ["subject_id", "target"])
check_required_columns(deployment_full, ["subject_id", "target"])

print("Train target distribution:")
display(train_full[TARGET_COL].value_counts(dropna=False).to_frame("count"))
display((train_full[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percent"))

print("\nTop missing-value columns in train_full:")
missing_summary = train_full.isna().sum().sort_values(ascending=False)
display(missing_summary.head(20))

Train target distribution:


,count
target,
0,372201
1,8260


,percent
target,
0,97.83
1,2.17



Top missing-value columns in train_full:


last_norm_lab             207472
bmi_last                  187776
last_diastolic            168450
last_pulse_pressure       168450
last_systolic             168450
weight_last               166688
max_prev_drg_mortality     32118
avg_prev_drg_mortality     32118
max_prev_drg_severity      32118
avg_prev_drg_severity      32118
num_unique_drugs               0
num_prev_transfers             0
num_prev_icu_visits            0
num_prev_infections            0
num_unique_organisms           0
num_resistant_cases            0
subject_id                     0
admission_type_urgent          0
admission_type_other           0
marital_status_single          0
dtype: int64

### **Create Validation Split**

In [7]:
groups = train_full["subject_id"].copy()
y_full = train_full[TARGET_COL].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(gss.split(train_full, y_full, groups=groups))

train_sub = train_full.iloc[train_idx].reset_index(drop=True)
val_df_full = train_full.iloc[val_idx].reset_index(drop=True)

print("train_sub:", train_sub.shape)
print("val_df_full:", val_df_full.shape)

train_subjects = set(train_sub["subject_id"].unique())
val_subjects = set(val_df_full["subject_id"].unique())
overlap = train_subjects.intersection(val_subjects)

print("Subject overlap between train_sub and val_df_full:", len(overlap))

train_sub: (303600, 54)
val_df_full: (76861, 54)
Subject overlap between train_sub and val_df_full: 0


### **Build Feature Matrices and Targets**

In [8]:
feature_cols = [c for c in train_full.columns if c not in DROP_FEATURE_COLS]

X_train = train_sub[feature_cols].copy()
y_train = train_sub[TARGET_COL].copy()

X_val = val_df_full[feature_cols].copy()
y_val = val_df_full[TARGET_COL].copy()

X_test = test_full[feature_cols].copy()
y_test = test_full[TARGET_COL].copy()

X_deployment = deployment_full[feature_cols].copy()
y_deployment = deployment_full[TARGET_COL].copy()

print("Number of model features:", len(feature_cols))
print(feature_cols[:30])
print()
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("X_deployment:", X_deployment.shape)

Number of model features: 49
['age', 'gender', 'ed_los_hours', 'came_from_ed', 'admission_hour', 'admission_day_of_week', 'num_prev_visits', 'time_since_last_visit_days', 'avg_time_between_visits_days', 'avg_prev_los', 'max_prev_los', 'last_los', 'avg_prev_ed_los', 'max_prev_ed_los', 'num_prev_diagnoses', 'num_unique_diagnoses_icd_codes', 'num_prev_procedures', 'num_unique_procedure_icd_codes', 'avg_prev_drg_severity', 'max_prev_drg_severity', 'avg_prev_drg_mortality', 'max_prev_drg_mortality', 'num_abnormal_labs', 'abnormal_lab_ratio', 'stat_lab_ratio', 'last_lab_flag', 'last_norm_lab', 'num_prev_medications', 'num_unique_drugs', 'num_prev_transfers']

X_train: (303600, 49)
X_val: (76861, 49)
X_test: (83356, 49)
X_deployment: (82211, 49)


### **Build AutoML Training Frames**

In [9]:
train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values

val_df = X_val.copy()
val_df[TARGET_COL] = y_val.values

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)

train_df: (303600, 50)
val_df: (76861, 50)


### **Define AutoML Settings**

In [10]:
if RUN_MODE == "core":
    fit_kwargs = dict(
        presets="medium",
        time_limit=1800,
        num_bag_folds=0,
        num_stack_levels=0,
        fit_weighted_ensemble=False,
        calibrate=False,
        hyperparameters={"RF": [{}], "XT": [{}]},
        num_cpus=2,
        save_space=True,
        keep_only_best=True,
        raise_on_model_failure=False,
        raise_on_no_models_fitted=False
    )
elif RUN_MODE == "full":
    fit_kwargs = dict(
        presets="best_quality",
        time_limit=3600,
        calibrate=False
    )
else:
    raise ValueError("RUN_MODE must be 'core' or 'full'")

fit_kwargs

{'presets': 'medium',
 'time_limit': 1800,
 'num_bag_folds': 0,
 'num_stack_levels': 0,
 'fit_weighted_ensemble': False,
 'calibrate': False,
 'hyperparameters': {'RF': [{}], 'XT': [{}]},
 'num_cpus': 2,
 'save_space': True,
 'keep_only_best': True,
 'raise_on_model_failure': False,
 'raise_on_no_models_fitted': False}

### **Train AutoML**

In [11]:
if SAVE_PATH.exists():
    shutil.rmtree(SAVE_PATH)

predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type="binary",
    eval_metric="average_precision",
    path=str(SAVE_PATH),
    verbosity=2
)

predictor.fit(
    train_data=train_df,
    tuning_data=val_df,
    **fit_kwargs
)

Preset alias specified: 'medium' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jan 19 22:01:41 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T8132
CPU Count:          10
Pytorch Version:    2.11.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.47 GB / 16.00 GB (21.7%)
Disk Space Avail:   227.70 GB / 460.43 GB (49.5%)
Presets specified: ['medium']
Failed to save metadata file due to exception 'NoneType' object has no attribute 'lower', skipping...
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/sarrachouk/Desktop/ehr-mortality-prediction/models/automl/autogluon_raw_core"
Train Data Rows:    303600
Train Data

### **Inspect AutoML Summary**

In [12]:
print("Best AutoML model by PR-AUC:", predictor.model_best)
print("Problem type:", predictor.problem_type)
print("Evaluation metric:", predictor.eval_metric)

Best AutoML model by PR-AUC: RandomForest
Problem type: binary
Evaluation metric: average_precision


In [13]:
leaderboard_val = predictor.leaderboard(val_df, silent=True)
display(leaderboard_val)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,0.133031,0.133031,average_precision,0.874434,0.667094,54.445817,0.874434,0.667094,54.445817,1,True,1


## **Re-rank Top Candidate Models Using:**
### PR-AUC -> Recall -> F1

In [14]:
top_n = min(5, len(leaderboard_val))
top_models = leaderboard_val["model"].head(top_n).tolist()

candidate_rows = []

for model_name in top_models:
    val_probs_model = get_positive_class_proba(predictor, X_val, model=model_name)
    temp_threshold_df = threshold_search_table(y_val, val_probs_model)
    temp_best_threshold, _ = choose_threshold_from_validation(
        temp_threshold_df,
        min_precision=BUSINESS_MIN_PRECISION
    )
    temp_metrics = compute_binary_metrics(y_val, val_probs_model, threshold=temp_best_threshold)
    temp_metrics["model"] = model_name
    candidate_rows.append(temp_metrics)

candidate_results_df = pd.DataFrame(candidate_rows)
candidate_results_ranked = rank_candidate_models(candidate_results_df)

display(candidate_results_ranked)

,pr_auc,roc_auc,recall,f1,precision,accuracy,threshold,model
0,0.133031,0.828397,0.909311,0.078591,0.04107,0.541172,0.01,RandomForest


In [15]:
final_model_name = candidate_results_ranked.iloc[0]["model"]
print("Final selected model using PR-AUC -> Recall -> F1:", final_model_name)

Final selected model using PR-AUC -> Recall -> F1: RandomForest


### **Select Threshold on Validation for the Final Selected Model**

In [19]:
val_probs = get_positive_class_proba(predictor, X_val, model=final_model_name)
val_threshold_df = threshold_search_table(y_val, val_probs)
val_pr_auc = average_precision_score(y_val, val_probs)

ranked_thresholds = val_threshold_df.copy()

if BUSINESS_MIN_PRECISION is not None:
    filtered = ranked_thresholds[ranked_thresholds["precision"] >= BUSINESS_MIN_PRECISION].copy()
    if len(filtered) > 0:
        ranked_thresholds = filtered

ranked_thresholds = ranked_thresholds.sort_values(
    by=["f1", "precision", "recall", "threshold"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

best_threshold = ranked_thresholds.loc[0, "threshold"]
best_threshold_row = ranked_thresholds.head(1).copy()
best_threshold_row["pr_auc_overall"] = val_pr_auc
best_threshold_row["selection_rule"] = "best F1, then precision, then recall"

ranked_thresholds = ranked_thresholds.copy()
ranked_thresholds["pr_auc_overall"] = val_pr_auc

print("Validation PR-AUC:", round(val_pr_auc, 6))
print("Balanced chosen threshold:", best_threshold)
display(best_threshold_row)
print("Top threshold candidates:")
display(ranked_thresholds.head(20))

Validation PR-AUC: 0.133031
Balanced chosen threshold: 0.13


,threshold,recall,f1,precision,accuracy,pr_auc_overall,selection_rule
0,0.13,0.258162,0.213767,0.182401,0.959134,0.133031,"best F1, then precision, then recall"


Top threshold candidates:


,threshold,recall,f1,precision,accuracy,pr_auc_overall
0,0.13,0.258162,0.213767,0.182401,0.959134,0.133031
1,0.14,0.238815,0.209438,0.186497,0.961203,0.133031
2,0.12,0.275695,0.208983,0.168266,0.955088,0.133031
3,0.11,0.301693,0.204215,0.154346,0.949402,0.133031
4,0.15,0.211608,0.203252,0.195531,0.964299,0.133031
5,0.16,0.191657,0.201975,0.213468,0.967409,0.133031
6,0.10,0.356106,0.201299,0.140305,0.939189,0.133031
7,0.17,0.180774,0.199134,0.221646,0.968710,0.133031
8,0.18,0.165054,0.191579,0.228261,0.970024,0.133031
9,0.09,0.388755,0.189759,0.125512,0.928559,0.133031


In [17]:
val_metrics, val_probs = evaluate_split(
    predictor, X_val, y_val, best_threshold, split_name="validation", model=final_model_name
)

test_metrics, test_probs = evaluate_split(
    predictor, X_test, y_test, best_threshold, split_name="test", model=final_model_name
)

deployment_metrics, deployment_probs = evaluate_split(
    predictor, X_deployment, y_deployment, best_threshold, split_name="deployment", model=final_model_name
)

results_df = pd.DataFrame([val_metrics, test_metrics, deployment_metrics])
display(results_df)

,pr_auc,roc_auc,recall,f1,precision,accuracy,threshold,split,model
0,0.133031,0.828397,0.909311,0.078591,0.041070,0.541172,0.01,validation,RandomForest
1,0.135581,0.825226,0.912429,0.077901,0.040687,0.541329,0.01,test,RandomForest
2,0.140762,0.834218,0.919819,0.080423,0.042050,0.546861,0.01,deployment,RandomForest


In [18]:
plot_pr_curve_plotly(y_val, val_probs, "Validation Precision-Recall Curve")
plot_roc_curve_plotly(y_val, val_probs, "Validation ROC Curve")
show_confusion_plotly(y_val, val_probs, best_threshold, f"Validation Confusion Matrix @ threshold={best_threshold:.2f}")